# Eddy heat flux (`v'T'` and `v'theta'`) for long-run experiments

This notebook computes daily zonal-mean eddy heat flux for the four long-run experiments:

- `B2000WCN001002_timefixed`
- `B2000WCN_NOCOUPL001002_timefixed`
- `B2000WCN007009010011_Clim3D_timefixed`
- `BWCN`

The output contains both:

\[
\overline{v'T'} = \overline{VT} - \overline{V}\,\overline{T}
\]

and

\[
\overline{v'\theta'} = \overline{V\theta} - \overline{V}\,\overline{\theta},
\qquad
\theta = T \left(\frac{100000\ \mathrm{Pa}}{p}\right)^{R_d/c_p}.
\]

Both quantities are first computed on CAM hybrid mid-levels, then interpolated to the standard pressure grid using log-pressure interpolation with zonal-mean hybrid mid-level pressure.

NaN-filled breakpoint days are preserved: if `V/T/PS` are all NaN on a day, both output variables for that day are NaN.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import gc
import re
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm

# ============================================================
# Configuration
# ============================================================

BASE_DIR = Path('/mnt/soclim0/public_data/weiji')

CASES = {
    'B2000WCN001002': BASE_DIR / 'B2000WCN001002_timefixed',
    'B2000WCN_NOCOUPL001002': BASE_DIR / 'B2000WCN_NOCOUPL001002_timefixed',
    'B2000WCN007009010011_Clim3D': BASE_DIR / 'B2000WCN007009010011_Clim3D_timefixed',
    'BWCN': BASE_DIR / 'BWCN',
}

# Same standard pressure grid used by Climatology/Epflux notebooks. Units: Pa.
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=np.float64)

DEBUG_MODE = False
DEBUG_CASES = ['B2000WCN001002']
DEBUG_YEARS = [1]

OVERWRITE = False
MAX_WORKERS = 4
TIME_CHUNK_DAYS = 31
VALID_ABS_MAX = 1.0e20
KAPPA_DRY_AIR = 287.0 / 1004.0
P0_THETA_PA = 100000.0

OUTPUT_SUBDIR = 'Eddyheatflux_daily_debug' if DEBUG_MODE else 'Eddyheatflux_daily'
CASES_TO_RUN = DEBUG_CASES if DEBUG_MODE else list(CASES)
YEARS_TO_RUN = DEBUG_YEARS if DEBUG_MODE else None

NETCDF_ENGINE = 'netcdf4'
COMPLEVEL = 1
YEAR_RE = re.compile(r'\.cam\.h3\.(\d{4})\.')

print('DEBUG_MODE =', DEBUG_MODE)
print('OUTPUT_SUBDIR =', OUTPUT_SUBDIR)
print('CASES_TO_RUN =', CASES_TO_RUN)
print('YEARS_TO_RUN =', YEARS_TO_RUN)
print('MAX_WORKERS =', MAX_WORKERS)
print('TIME_CHUNK_DAYS =', TIME_CHUNK_DAYS)


In [ ]:
# ============================================================
# Helpers: files, pressure, interpolation
# ============================================================

def parse_year(path):
    m = YEAR_RE.search(Path(path).name)
    if m is None:
        return None
    return int(m.group(1))


def discover_var_files(case_root, var, years=None):
    files = sorted((Path(case_root) / var).glob(f'*.{var}.nc'), key=lambda p: (parse_year(p) or 10**9, p.name))
    if years is not None:
        year_set = {int(y) for y in years}
        files = [p for p in files if parse_year(p) in year_set]
    return files


def discover_year_inputs(case_root, years=None):
    v_by_year = {parse_year(p): p for p in discover_var_files(case_root, 'V', years=years) if parse_year(p) is not None}
    t_by_year = {parse_year(p): p for p in discover_var_files(case_root, 'T', years=years) if parse_year(p) is not None}
    ps_by_year = {parse_year(p): p for p in discover_var_files(case_root, 'PS', years=years) if parse_year(p) is not None}

    common_years = sorted(set(v_by_year) & set(t_by_year))
    missing_t = sorted(set(v_by_year) - set(t_by_year))
    missing_v = sorted(set(t_by_year) - set(v_by_year))
    pairs = [(year, v_by_year[year], t_by_year[year], ps_by_year.get(year)) for year in common_years]
    return pairs, missing_v, missing_t


def _open_dataset(path):
    try:
        return xr.open_dataset(path, decode_times=False, engine=NETCDF_ENGINE)
    except Exception:
        return xr.open_dataset(path, decode_times=False)


def interp_profile_logp_3d(v_hyb, p_hyb, p_tgt_pa):
    """
    Log-pressure interpolation for fields shaped (time, lat, lev).
    Returns (time, lat, plev); caller can transpose to (time, plev, lat).
    """
    p_tgt_pa = np.asarray(p_tgt_pa, dtype=np.float64)

    def _interp_col(vcol, pcol):
        vcol = np.asarray(vcol, dtype=np.float64)
        pcol = np.asarray(pcol, dtype=np.float64)
        valid = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0.0)
        if valid.sum() < 2:
            return np.full(p_tgt_pa.shape, np.nan, dtype=np.float64)

        p_use = pcol[valid]
        v_use = vcol[valid]
        order = np.argsort(p_use)
        p_use = p_use[order]
        v_use = v_use[order]

        return np.interp(
            np.log(p_tgt_pa),
            np.log(p_use),
            v_use,
            left=np.nan,
            right=np.nan,
        )

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb,
        p_hyb,
        input_core_dims=[['lev'], ['lev']],
        output_core_dims=[['plev']],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={'output_sizes': {'plev': len(p_tgt_pa)}},
    )
    return out.assign_coords(plev=('plev', p_tgt_pa))


def output_file_for(case_root, case_name, nyrs):
    return Path(case_root) / OUTPUT_SUBDIR / f'EHF_{case_name}_vTprime_vThetaprime_{nyrs}yr_time_plev_lat.nc'


def summary_file_for(case_root, case_name):
    return Path(case_root) / OUTPUT_SUBDIR / f'EHF_{case_name}_processing_summary.csv'


def safe_time_attrs(ds):
    return ds['time'].attrs.copy() if 'time' in ds else {}


def make_encoding(ds):
    encoding = {}
    for name, da in ds.data_vars.items():
        enc = {'zlib': True, 'complevel': COMPLEVEL}
        if np.issubdtype(da.dtype, np.floating):
            enc.update({'dtype': 'float32', '_FillValue': np.float32(np.nan)})
        encoding[name] = enc
    encoding['plev'] = {'dtype': 'float64'}
    return encoding


def save_dataset_atomic(ds, out_file):
    out_file = Path(out_file)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE:
        print(f'[SKIP] existing: {out_file}')
        return out_file

    tmp = out_file.with_name(out_file.name + '.tmp')
    if tmp.exists():
        tmp.unlink()

    print(f'[WRITE] {out_file}')
    ds.to_netcdf(tmp, engine=NETCDF_ENGINE, format='NETCDF4', encoding=make_encoding(ds))
    tmp.replace(out_file)
    return out_file


In [ ]:
# ============================================================
# Per-year EHF calculation
# ============================================================

def compute_ehf_chunk(ds_v, ds_t, ps_da, start, stop):
    time_slice = slice(start, stop)

    V = ds_v['V'].isel(time=time_slice)
    T = ds_t['T'].isel(time=time_slice)

    V = V.where(np.abs(V) < VALID_ABS_MAX)
    T = T.where(np.abs(T) < VALID_ABS_MAX)

    ps = ps_da.isel(time=time_slice).where(np.abs(ps_da.isel(time=time_slice)) < VALID_ABS_MAX)
    p0_model = ds_v['P0'] if 'P0' in ds_v else xr.DataArray(P0_THETA_PA)
    p_mid = (ds_v['hyam'] * p0_model + ds_v['hybm'] * ps).transpose(*T.dims)
    p_mid = p_mid.where(np.isfinite(p_mid) & (p_mid > 0.0))

    theta = T * (P0_THETA_PA / p_mid) ** KAPPA_DRY_AIR
    theta = theta.where(np.abs(theta) < VALID_ABS_MAX)

    # Pairwise valid mask keeps partial missing longitudes from biasing the covariance.
    valid_pair = V.notnull() & T.notnull() & theta.notnull() & p_mid.notnull()
    Vp_in = V.where(valid_pair)
    Tp_in = T.where(valid_pair)
    Thetap_in = theta.where(valid_pair)

    V_zm = Vp_in.mean('lon', skipna=True)
    T_zm = Tp_in.mean('lon', skipna=True)
    Theta_zm = Thetap_in.mean('lon', skipna=True)
    VT_zm = (Vp_in * Tp_in).mean('lon', skipna=True)
    VTheta_zm = (Vp_in * Thetap_in).mean('lon', skipna=True)

    vt_hyb = (VT_zm - V_zm * T_zm).transpose('time', 'lat', 'lev')
    vt_hyb.name = 'EHF_vTprime_hybrid'

    vtheta_hyb = (VTheta_zm - V_zm * Theta_zm).transpose('time', 'lat', 'lev')
    vtheta_hyb.name = 'EHF_vThetaprime_hybrid'

    p_zm = p_mid.mean('lon', skipna=True).transpose('time', 'lat', 'lev')

    vt_std = interp_profile_logp_3d(vt_hyb, p_zm, PLEV_STD_PA).transpose('time', 'plev', 'lat')
    vt_std.name = 'EHF_vTprime'
    vt_std.attrs.update({
        'long_name': "Daily zonal-mean eddy heat flux v'T'",
        'units': 'K m s-1',
        'definition': "[V*T] - [V][T], computed on hybrid levels then log-p interpolated",
    })

    vtheta_std = interp_profile_logp_3d(vtheta_hyb, p_zm, PLEV_STD_PA).transpose('time', 'plev', 'lat')
    vtheta_std.name = 'EHF_vThetaprime'
    vtheta_std.attrs.update({
        'long_name': "Daily zonal-mean eddy heat flux v'theta'",
        'units': 'K m s-1',
        'definition': "[V*theta] - [V][theta], theta = T*(100000 Pa/p_mid)^(R_d/c_p), computed on hybrid levels then log-p interpolated",
        'theta_reference_pressure_pa': P0_THETA_PA,
        'kappa': KAPPA_DRY_AIR,
    })

    return xr.Dataset({
        'EHF_vTprime': vt_std.astype(np.float32),
        'EHF_vThetaprime': vtheta_std.astype(np.float32),
    }).load()


def process_one_year(args):
    case_name, year, v_file, t_file, ps_file = args

    rec = {
        'case': case_name,
        'year': int(year),
        'v_file': str(v_file),
        't_file': str(t_file),
        'ps_file': str(ps_file) if ps_file is not None else '',
        'status': 'ok',
        'message': '',
    }

    ds_v = None
    ds_t = None
    ds_ps = None
    try:
        ds_v = _open_dataset(v_file)
        ds_t = _open_dataset(t_file)

        for var, ds in [('V', ds_v), ('T', ds_t)]:
            required = {var, 'hyam', 'hybm'}
            missing = required - set(ds.variables)
            if missing:
                raise ValueError(f'{Path(v_file if var == "V" else t_file).name} missing {sorted(missing)}')

        if ds_v.sizes.get('time', 0) != ds_t.sizes.get('time', 0):
            raise ValueError(f'time length mismatch: V={ds_v.sizes.get("time")} T={ds_t.sizes.get("time")}')
        if not np.array_equal(ds_v['time'].values, ds_t['time'].values):
            raise ValueError('time coordinate mismatch between V and T files')

        if 'PS' in ds_v:
            ps_da = ds_v['PS']
        elif ps_file is not None:
            ds_ps = _open_dataset(ps_file)
            if 'PS' not in ds_ps:
                raise ValueError(f'{Path(ps_file).name} missing PS')
            ps_da = ds_ps['PS']
        else:
            raise ValueError(f'{Path(v_file).name} has no embedded PS and no paired PS file was found')

        ntime = int(ds_v.sizes['time'])
        if ntime == 0:
            raise ValueError('empty time dimension')

        chunks = []
        for start in range(0, ntime, TIME_CHUNK_DAYS):
            stop = min(start + TIME_CHUNK_DAYS, ntime)
            chunks.append(compute_ehf_chunk(ds_v, ds_t, ps_da, start, stop))
            gc.collect()

        ds_out = xr.concat(chunks, dim='time').transpose('time', 'plev', 'lat')
        ds_out['plev'].attrs.update({'long_name': 'pressure', 'units': 'Pa', 'positive': 'down'})
        ds_out['lat'].attrs.update(ds_v['lat'].attrs)
        ds_out['time'].attrs.update(safe_time_attrs(ds_v))

        if 'date' in ds_v:
            ds_out = ds_out.assign_coords(date=('time', ds_v['date'].values.astype(np.int32)))
            ds_out['date'].attrs.update(ds_v['date'].attrs)
        if 'datesec' in ds_v:
            ds_out = ds_out.assign_coords(datesec=('time', ds_v['datesec'].values.astype(np.int32)))
            ds_out['datesec'].attrs.update(ds_v['datesec'].attrs)

        vt_arr = ds_out['EHF_vTprime'].values
        vtheta_arr = ds_out['EHF_vThetaprime'].values
        rec['ntime'] = int(ntime)
        rec['finite_fraction_vTprime'] = float(np.isfinite(vt_arr).sum() / vt_arr.size) if vt_arr.size else np.nan
        rec['finite_fraction_vThetaprime'] = float(np.isfinite(vtheta_arr).sum() / vtheta_arr.size) if vtheta_arr.size else np.nan
        rec['n_all_nan_times_vTprime'] = int(np.isnan(vt_arr).all(axis=(1, 2)).sum()) if vt_arr.ndim == 3 else np.nan
        rec['n_all_nan_times_vThetaprime'] = int(np.isnan(vtheta_arr).all(axis=(1, 2)).sum()) if vtheta_arr.ndim == 3 else np.nan

        return ds_out, rec

    except Exception as exc:
        rec['status'] = 'error'
        rec['message'] = f'{type(exc).__name__}: {exc}'
        return None, rec
    finally:
        for ds in (ds_v, ds_t, ds_ps):
            if ds is not None:
                ds.close()
        gc.collect()


In [ ]:
# ============================================================
# Case driver
# ============================================================

def build_case_attrs(case_name, case_root, records):
    ok = [r for r in records if r.get('status') == 'ok']
    return {
        'title': "Daily zonal-mean eddy heat flux v'T' and v'theta'",
        'case_name': case_name,
        'case_root': str(case_root),
        'source_variables': 'V, T, PS, hyam, hybm, P0',
        'definition': "EHF_vTprime = [V*T] - [V][T]; EHF_vThetaprime = [V*theta] - [V][theta]",
        'theta_definition': "theta = T*(100000 Pa/p_mid)^(R_d/c_p), kappa=287/1004",
        'method': "Compute both covariances on CAM hybrid mid-levels, then interpolate to pressure levels using zonal-mean hyam*P0 + hybm*PS",
        'pressure_interpolation': 'linear interpolation in log(pressure)',
        'pressure_units': 'Pa',
        'plev_values_pa': ','.join(str(int(p)) for p in PLEV_STD_PA),
        'nan_policy': 'NaN-filled days remain NaN; longitude means use pairwise valid V/T/theta points.',
        'source_file_count_ok': len(ok),
        'debug_mode': str(DEBUG_MODE),
    }


def process_case(case_name):
    root = Path(CASES[case_name])
    print('\n' + '=' * 100)
    print(f'CASE: {case_name}')
    print(f'ROOT: {root}')
    print('=' * 100)

    if not root.exists():
        raise FileNotFoundError(f'Missing case root: {root}')

    inputs, missing_v, missing_t = discover_year_inputs(root, years=YEARS_TO_RUN)
    if missing_v:
        print(f'[WARN] years with T but no V: {missing_v[:10]}')
    if missing_t:
        print(f'[WARN] years with V but no T: {missing_t[:10]}')
    if not inputs:
        raise FileNotFoundError(f'No paired V/T annual files found under {root}')

    years = [x[0] for x in inputs]
    nyrs = len(years)
    out_file = output_file_for(root, case_name, nyrs)
    summary_file = summary_file_for(root, case_name)

    if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE:
        print(f'[SKIP] existing output: {out_file}')
        return out_file

    print(f'years: n={nyrs}, first={years[0]}, last={years[-1]}')
    print(f'output: {out_file}')

    args_list = [(case_name, year, v_file, t_file, ps_file) for year, v_file, t_file, ps_file in inputs]
    datasets = []
    records = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(process_one_year, args): args[1] for args in args_list}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f'EHF {case_name}', unit='year'):
            ds_year, rec = fut.result()
            records.append(rec)
            if ds_year is not None:
                datasets.append(ds_year)
            else:
                print(f"[ERROR] {case_name} year {rec.get('year')}: {rec.get('message')}")

    records = sorted(records, key=lambda r: int(r.get('year', 10**9)))
    summary_file.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(records).to_csv(summary_file, index=False)
    print(f'[WRITE] {summary_file}')

    if not datasets:
        raise RuntimeError(f'{case_name}: no yearly datasets were computed successfully')

    ds_full = xr.concat(datasets, dim='time').sortby('time')
    tvals = ds_full['time'].values
    _, idx = np.unique(tvals, return_index=True)
    ds_full = ds_full.isel(time=np.sort(idx))
    ds_full.attrs.update(build_case_attrs(case_name, root, records))

    written = save_dataset_atomic(ds_full, out_file)

    ds_full.close()
    for ds in datasets:
        ds.close()
    gc.collect()
    return written


In [ ]:
# ============================================================
# Run all requested cases
# ============================================================

all_outputs = {}
for case_name in tqdm(CASES_TO_RUN, desc='Cases', unit='case'):
    all_outputs[case_name] = process_case(case_name)

print('\nDone. Outputs:')
for case_name, out in all_outputs.items():
    print(case_name)
    print('  ', out)
